# STEM Digital Twin — what the twin is, and what sits beside it

The twin is a **kinematical** STEM simulator with a vendor-neutral instrument-control
interface. That sentence is the whole scope, and v5 spent most of its effort making it
true by *removing* things rather than adding them.

## The twin

Samples, imaging, kinematical diffraction, thickness selection, drift, contamination,
depth of field, feature-finding, magnification and stage safety, and the backend
abstraction. Fast (seconds), no heavy dependencies, runs on CPU in Colab.

One interface underpins all of it:

```python
positions_A, Z = sample.get_atoms_in_region(cx_um, cy_um, half_width_um, depth_nm)
```

Anything that needs the specimen's atoms goes through that method, so a change to a
sample is seen everywhere at once. `project_with_dof` is the same idea one level down for
focus: it reads the `(D, H, W)` voxel volume every sample produces, so depth of field
reaches all thirteen samples with no per-sample physics.

## Setting up conditions (v6)

There are no named environment presets. A demo states its conditions as numbers:

```python
sim.set_drift(vx_nm_per_s=1.2, vy_nm_per_s=0.7, enabled=True)
sim.set_contamination(enabled=True, rate=100)      # rate is a percentage: 100 = nominal
sim.set_noise(dwell_us=15.0, dqe=0.8, readout_e=1.5)
control.set_autofocus_limits(min_contrast=0.10)
```

A preset name told you nothing about what was actually set. These four calls are the whole
surface, and what you read at the call site is what the twin is set to.

## Tilt is a rigid rotation (v6)

`project_with_dof` rotates the specimen about Y by beta then about X by alpha and integrates
along the lab beam axis. Earlier builds sheared each depth slice by `tan(angle) * 0.35`,
which was **10.9x too strong** and had no foreshortening at all. Foreshortening now tracks
cos(beta) to ~1%. Because the rotation gives the true beam distance, depth of field needs no
separate tilt term — specimen depth, uniform defocus and the tilted focal plane are one
expression.

## Roaming specimen (v6)

**Every sample roams.** Stage moves and drift both work in absolute world coordinates, so
neither hits a wall and the two agree at the boundary (earlier builds had stage motion
*wrap* while drift *clamped*).

- `roaming_mode = "periodic"` (12 samples, the default): the volume is one tile of a
  repeating specimen and the sampler wraps. Costs nothing. The specimen repeats every
  `generation_range_um` — 20 µm by default, about 5.5 hours at 1 nm/s drift.
- `roaming_mode = "world"` (`shape_assembly`): generated from a position hash, so the server
  re-tiles and the specimen **never** repeats. Driving back reveals what you left, because
  it was never stored.

## What was removed in v5, and why

| Removed | Why |
|---|---|
| **Beam damage** | A contrast-decay curve with no spatial story. Contamination does the same pedagogical job *and* leaves a footprint a workflow has to navigate around. |
| **EELS** | The spectrum was a structured dummy. Leaving it in the twin invited it to be read as a physics model. → `Appendix_B_EELS` |
| **abTEM / py4DSTEM** | Real physics, wrong place. Slow, GPU-hungry, and it pins NumPy versions. → `Appendix_A_abTEM_Multislice` |

Contamination was also **recalibrated**: `contamination_rate` is now a percentage knob
with 100 as the nominal default, and the internal constants were fixed. They had been
roughly 1000× too slow, which is why the footprint demos never showed a footprint.

## The appendices

Each states that a capability *can* be added and shows it working, without the twin
depending on it. None is part of GridScope.

| Notebook | What it shows | Where it plugs in |
|---|---|---|
| `Appendix_A_abTEM_Multislice` | Dynamical diffraction, 4D-STEM, py4DSTEM round-trip | `atoms_from_twin_sample` → `get_atoms_in_region` |
| `Appendix_B_EELS` | Single-spot spectrum acquisition and the API surface | monkey-patched `acquire_spectrum`; a vendor backend on hardware |
| `Appendix_C_Portability_Backends` | The ten-method porting contract, written to hand to an LLM | `microscope_backend.py` vendor classes |
| `Appendix_D_Ambiguous_Workflow` | Why a language model earns its place | `feature_finding.py` |

C and D were sections 7 and 8 of the main notebook. They are the *argument* for the whole
project, and they were being read as implementation detail because of where they sat.

## Reading order

1. **`STEM_Digital_Twin_Kinematical_v5.ipynb`** — the twin. Start here; it is self-contained.

Every notebook writes the **same** `samples/` package, server and client — byte-identical,
verified by md5. They differ only in what they demonstrate:

| File | Function |
|---|---|
| `STEM_Digital_Twin_Kinematical_v5` | the twin + core demos |
| `Demo_Limits_and_Bounds_v5` | where parameters stop behaving: drift, focus, dose, thickness |
| `Demo_ShapeAssembly_v5` | everything about `shape_assembly`: structure params, depth behaviour, tilt band, and the analytic-projector fidelity reference (merged from the old demo + Appendix E) |
| `Appendix_A`…`Appendix_D` | capabilities deliberately outside the twin |

2. **`Appendix_C_Portability_Backends`** — why the twin is worth building.
3. **`Appendix_D_Ambiguous_Workflow`** — why an agent is worth putting on top of it.
4. **`Appendix_A` / `Appendix_B`** — what you would reach for when the kinematical
   approximation stops being enough.

## Real-microscope backends

`microscope_backend.py` holds the abstract `MicroscopeBackend` plus `TwinBackend`,
`ThermoFisherBackend`, `NionBackend`, `JEOLBackend`, all sharing one 10-method surface:
`get_stage`/`set_stage`, `get_beam`/`set_beam`, `set_mode`, `set_fov_um`,
`get_magnification`/`set_magnification`, `acquire_image`, `autofocus`.

The vendor classes are **honest skeletons** — real SDK calls in the bodies (AutoScript,
`nion.instrumentation`, PyJEM `TEM3`), **unvalidated on hardware**. To deploy, verify those
calls for your instrument and instantiate that backend instead of `TwinBackend`. The
beam-safety pattern (`set_beam(..., disabled=True)`) and the units (µm / degrees / pA / kV
/ magnification) are shared.

## Adding something new

| Extension | File to edit | What to add |
|---|---|---|
| Alt. multislice engine (Prismatic/GPU) | new module, mirror `AbtemDiffraction` | `saed`/`cbed`/`scan_4d` over your engine; same in/out |
| 4D-STEM analysis (py4DSTEM) | *no twin code* | consume a saved `.npy` in an isolated `numpy<2` env |
| Real instrument control | `microscope_backend.py` | fill vendor SDK calls |
| Real EELS | `microscope_backend.py` | add `acquire_spectrum` wrapping the spectrometer SDK |
| New sample from real atoms | `samples/atomsk_polycrystal.py` (pattern) | load a file, expose `get_atoms_in_region` |

See `STEM_Digital_Twin_and_GridScope_CHANGELOG.md` Parts 3 and 4 for the full old→new
breakdown with measured before/after numbers.
